In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


In [3]:
%pip show langchain-pinecone

Name: langchain-pinecone
Version: 0.2.13
Summary: An integration package connecting Pinecone and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: c:\users\admin\anaconda3\envs\llm\lib\site-packages
Requires: httpx, langchain-core, langchain-openai, numpy, pinecone, pydantic, simsimd
Required-by: 
Note: you may need to restart the kernel to use updated packages.


# 1. knowlege base 구성을 위한 데이터 생성

In [18]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 7.61 s
Wall time: 7.64 s


180

In [ ]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [19]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [20]:
len(embedding.embed_query("샘플"))

4096

In [9]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [21]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# 데이터를 처음 저장할 때
index_name = 'tax-index'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 15.6 ms
Wall time: 5.72 ms


In [22]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


# 2. 답변 생성을 위한 Retrieval

In [23]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [24]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n','\n') for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [25]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [26]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [27]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가이비다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [28]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

2023년 한국 소득세법에 따라 연봉 5,000만원인 직장인의 소득세를 계산해 드리겠습니다. 다음 단계를 참고해주세요.

### 1. **근로소득금액 산정**  
- 연봉 5,000만원에서 **근로소득공제** 적용  
  (5,000만원 × 15% + 180만원) = **930만원** 공제  
  → **과세표준** = 5,000만원 - 930만원 = **4,070만원**

### 2. **산출세액 계산 (누진세율 적용)**  
- 1,200만원 이하: 1,200만원 × 6% = 72만원  
- 1,200만원~4,070만원 (2,870만원): 2,870만원 × 15% = 430.5만원  
  → **산출세액** = 72 + 430.5 = **502.5만원**

### 3. **근로소득세액공제**  
- 502.5만원 × 55% (550만원 한도) = **276.375만원** 공제  
  → **세액** = 502.5 - 276.375 = **226.125만원**

### 4. **최종 납부세액**  
- 약 **226만원** (월평균 18.8만원)이지만, 실제 급여에서는 **4대 보험료(국민연금, 건강보험 등) 공제 후** 세액이 계산되므로 실제 납부액은 약간 적을 수 있습니다.

### 📌 참고 사항  
- 4대 보험료(국민연금, 건강보험, 고용보험, 장기요양보험)가 약 8~10% 추가 공제됩니다.  
- 지방소득세(소득세의 10%)가 별도로 부과되므로 총 납부세액은 약 **249만원** 수준입니다.  

예시)  
- 연봉 5,000만원 시 **실수령액**은 약 3,800만원~3,900만원 정도로 예상됩니다.  

(계산 시 공제항복(의료비, 교육비 등)이 없는 단순 기준입니다.)
